# 수집 결과 전처리 노트북

1. raw 데이터 읽기
- places.csv
- place_features.csv
- review_docs.csv
2. 공통 전처리 함수 정의
- 공백 정리
- address_dong 추출
- yes/no 표준화
- 리뷰 잡음 제거
- 나이 표현 추출
- 태그 추출
- chunk 분할
3. place_features.csv 전처리
- 컬럼이 없으면 빈 컬럼 추가
- yes/no/limited/easy 같은 값 표준화
- address_dong 보강
4. places.csv 전처리
- car_info가 있으면 parking_info에 병합 후 제거
- 문자열 컬럼 공백 정리
- 숫자 컬럼을 nullable integer로 변환
- 공식 정보 기반 추천 보조 컬럼 추가
5. review_docs.csv 전처리
- source_url 보정
- 리뷰 잡음 제거
- 문단 분리
- 다른 매장만 말하는 문단 제거
- 공식정보만 반복하는 문단 축소
- 중복 제거
- 태그 추출
- 긴 리뷰를 chunk로 분할
- 최종 chunk row 생성
6. preprocessed 3개 파일 저장
- places.csv
- place_features.csv
- review_docs.csv
### 입출력 경로
- 원본 입력: `outputs/raw/`
- 전처리 출력: `outputs/preprocessed/`

In [1]:
from __future__ import annotations

import hashlib
import re
from pathlib import Path
from typing import Iterable

import pandas as pd
import requests


def resolve_base_dir() -> Path:
    cwd = Path.cwd()
    candidates = [cwd, cwd / '01_data_prep', cwd.parent / '01_data_prep']
    for candidate in candidates:
        if (candidate / 'data_collection.ipynb').exists() and (candidate / 'preprocessing.ipynb').exists():
            return candidate
    return cwd


BASE_DIR = resolve_base_dir()
RAW_OUTPUT_DIR = BASE_DIR / 'outputs' / 'raw'
PREPROCESSED_OUTPUT_DIR = BASE_DIR / 'outputs' / 'preprocessed'
PREPROCESSED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PLACES_PATH = RAW_OUTPUT_DIR / 'places.csv'
RAW_REVIEW_DOCS_PATH = RAW_OUTPUT_DIR / 'review_docs.csv'
RAW_PLACE_FEATURES_PATH = RAW_OUTPUT_DIR / 'place_features.csv'

OUTPUT_PLACES_PATH = PREPROCESSED_OUTPUT_DIR / 'places.csv'
OUTPUT_REVIEW_DOCS_PATH = PREPROCESSED_OUTPUT_DIR / 'review_docs.csv'
OUTPUT_PLACE_FEATURES_PATH = PREPROCESSED_OUTPUT_DIR / 'place_features.csv'

SET_POINT_RE = re.compile(
    r'setPoint\("(?P<lat>[0-9.]+)"\s*,\s*"(?P<lng>[0-9.]+)"',
    flags=re.I,
)


def fetch_place_coordinates(session: requests.Session, url: object) -> tuple[float | None, float | None]:
    normalized = '' if url is None or pd.isna(url) else str(url).strip()
    if not normalized:
        return None, None
    response = session.get(normalized, timeout=20)
    response.raise_for_status()
    match = SET_POINT_RE.search(response.text)
    if not match:
        return None, None
    return float(match.group('lat')), float(match.group('lng'))


def enrich_place_coordinates(places_df: pd.DataFrame) -> pd.DataFrame:
    enriched = places_df.copy()
    if 'latitude' not in enriched.columns:
        enriched['latitude'] = pd.NA
    if 'longitude' not in enriched.columns:
        enriched['longitude'] = pd.NA

    session = requests.Session()
    session.headers.update({'User-Agent': 'BebeNori/1.0'})
    try:
        for idx, row in enriched.iterrows():
            lat = pd.to_numeric(pd.Series([row.get('latitude')]), errors='coerce').iloc[0]
            lng = pd.to_numeric(pd.Series([row.get('longitude')]), errors='coerce').iloc[0]
            if pd.notna(lat) and pd.notna(lng):
                continue

            fetched_lat, fetched_lng = fetch_place_coordinates(session, row.get('directions_url', ''))
            if fetched_lat is None or fetched_lng is None:
                continue

            enriched.at[idx, 'latitude'] = round(fetched_lat, 6)
            enriched.at[idx, 'longitude'] = round(fetched_lng, 6)
    finally:
        session.close()

    return enriched

# 네이버 블로그 리뷰는 보이지 않는 공백과 위젯 잡음이 많아서 전처리 규칙을 별도로 둡니다.
ZERO_WIDTH_CHARS = "\u200b\u200c\u200d\ufeff"
ZERO_WIDTH_RE = re.compile(f"[{ZERO_WIDTH_CHARS}]")
ZERO_WIDTH_CLUSTER_RE = re.compile(f"(?:\\s*[{ZERO_WIDTH_CHARS}]\\s*){{2,}}")
MAP_WIDGET_RE = re.compile(r"50m © NAVER Corp\..*?이 블로그의 체크인 이 장소의 다른 글", flags=re.S)
VIDEO_WIDGET_RE = re.compile(r"0초 0초 100% 광고 후 계속됩니다\..*?도움말 라이선스 디버그 정보 다운로드", flags=re.S)
INLINE_URL_RE = re.compile(r"https?://\S+")
INLINE_DOMAIN_RE = re.compile(r"\b(?:m\.)?blog\.naver\.com\b|\bumppa\.seoul\.go\.kr\b")
HASHTAG_RE = re.compile(r"(?<!\w)#[^\s#]+")
NON_ALNUM_RE = re.compile(r"[^가-힣A-Za-z0-9]")
MONTH_AGE_RE = re.compile(r"(?<![\d~\-])(\d{1,2})\s*개월(?!\s*[~\-])")
YEAR_AGE_RE = re.compile(r"(?<![\d~\-])(\d{1,2})\s*세(?!\s*[~\-])")

AGE_BAND_RULES = [
    (0, 12, '0-12개월'),
    (13, 24, '13-24개월'),
    (25, 36, '25-36개월'),
    (37, 48, '37-48개월'),
    (49, 60, '49-60개월'),
    (61, 72, '61-72개월'),
    (73, 120, '73개월+'),
]

BULLET_MARKERS = ['📍', '📌', '🔹', '🔸', '✔️', '✔', '✅', '▷', '▶', '👉', '•']

OFFICIAL_ONLY_KEYWORDS = ['이용연령', '이용시간', '운영일', '이용요금', '문의', '예약', '서울특별시', '위치', '주차']
EXPERIENCE_KEYWORDS = ['좋았', '좋아요', '좋더라', '잘 놀', '재밌', '재미', '불편', '편했', '편하', '수월', '무서', '지루', '만족', '붐비', '사람이 많']
EXTRA_NOISE_PHRASES = [
    '재생 (space/k)',
    '재생 음소거 (m)',
    '실시간 설정 전체 화면 (f)',
    '다음 동영상 subject author 취소',
    '3G/LTE 등으로 재생 시 데이터 사용료가 발생할 수 있습니다.',
    '죄송합니다. 문제가 발생했습니다. 다시 시도해 주세요.',
    '화면을 돌리거나 터치로 움직여 보세요',
    '해상도 자동 (480p)',
    '자막 사용 안함',
    '재생 속도 1.0x (기본)',
    '음소거 상태입니다.',
]

# 추천과 검색에서 함께 쓸 공통 태그 사전입니다.
PLAY_TAG_PATTERNS = {
    '감각놀이': [r'감각놀이', r'촉감', r'편백', r'모래놀이', r'물놀이', r'오감', r'치발기', r'말랑'],
    '대근육': [r'트램펄린', r'정글짐', r'클라이밍', r'그물', r'미끄럼틀', r'점프', r'챌린지', r'징검다리', r'몸 쓰는 놀이'],
    '소근육': [r'소근육', r'블록', r'레고', r'퍼즐', r'보드게임', r'구슬', r'끼우', r'쌓'],
    '역할놀이': [r'역할놀이', r'주방놀이', r'병원놀이', r'마켓', r'캠핑존', r'인형', r'의상', r'전통옷'],
    '창의활동': [r'미술', r'만들기', r'색칠', r'그림', r'드로잉', r'아뜰리에', r'공예', r'창의'],
    '사회성놀이': [r'친구와', r'함께', r'차례', r'협동', r'줄다리기', r'단체'],
    '미디어체험': [r'디지털', r'미디어', r'인터랙티브', r'라이브스케치', r'스크린'],
    '낚시놀이': [r'낚시놀이', r'낚시'],
}

NEED_TAG_PATTERNS = {
    '주차편의': [r'주차.*(수월|편|괜찮|여유|널널)', r'주차 가능', r'입구가 코앞'],
    '대중교통편의': [r'지하철 바로 앞', r'도보 약', r'역.*가깝', r'접근성이 정말 좋'],
    '예약치열': [r'오픈런', r'당일 예약 거의 불가', r'예약이 힘들', r'주말은.*예약'],
    '예약여유': [r'예약.*널널', r'취소자리', r'전날에도 충분'],
    '보호자휴식': [r'보호자.*앉', r'엄마들.*앉', r'숨 돌릴', r'쉬는 공간'],
    '청결': [r'깨끗', r'청결', r'시설 관리가 잘'],
    '넓음': [r'넓', r'개방감', r'공간자체가 엄청 넓'],
    '형제동반': [r'형제', r'자매', r'다둥이'],
}

ISSUE_TAG_PATTERNS = {
    '혼잡주의': [r'사람이 많', r'붐비', r'복잡', r'치이', r'기다리'],
    '주차주의': [r'주차.*(협소|어렵|빡세|불편|힘들)', r'인근 주차장'],
    '접근성주의': [r'대중교통.*힘들', r'오르막', r'내리막', r'입구.*찾', r'엘베는 없', r'계단'],
    '연령주의': [r'너무 어리', r'무서', r'큰 아이', r'험하게', r'지루해 보', r'비추천'],
    '안전주의': [r'스릴', r'가파르', r'넘어지', r'치임', r'위험', r'거칠'],
}


def normalize_space(value: object) -> str:
    if value is None or pd.isna(value):
        return ''
    text = str(value).replace('\xa0', ' ').replace('\r\n', '\n').replace('\r', '\n')
    text = ZERO_WIDTH_RE.sub(' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def compact_text(value: object) -> str:
    return NON_ALNUM_RE.sub('', normalize_space(value))


def join_unique_pipe(values: Iterable[str]) -> str:
    normalized_values = [normalize_space(value) for value in values]
    return '|'.join(value for value in dict.fromkeys(normalized_values) if value)


def merge_unique_text(left: object, right: object) -> str:
    return join_unique_pipe([left, right]).replace('|', ' ')


def extract_address_dong(value: object) -> str:
    normalized = normalize_space(value)
    for parenthetical_text in re.findall(r'\(([^)]*)\)', normalized):
        match = re.search(r'([가-힣A-Za-z0-9]+동)', parenthetical_text)
        if match:
            return match.group(1)
    return ''


def extract_source_domain(url: object) -> str:
    normalized = normalize_space(url)
    match = re.search(r'https?://([^/]+)', normalized)
    return match.group(1) if match else ''


def normalize_feature_value(column_name: str, value: object) -> str:
    normalized = normalize_space(value).lower()
    if not normalized:
        return ''
    if column_name in {'cleanliness_positive', 'spacious_positive'}:
        return 'yes' if normalized in {'mentioned', 'yes', 'y', 'true', '1'} else normalized
    if column_name in {'crowded_warning', 'safety_negative'}:
        return 'yes' if normalized in {'warning', 'yes', 'y', 'true', '1'} else normalized
    if column_name == 'parking_difficulty' and normalized == 'normal':
        return 'easy'
    if normalized in {'yes', 'y', 'true', '1'}:
        return 'yes'
    if normalized in {'no', 'n', 'false', '0'}:
        return 'no'
    return normalized


def age_band_from_months(months: int | None) -> str:
    if months is None:
        return ''
    for min_month, max_month, label in AGE_BAND_RULES:
        if min_month <= months <= max_month:
            return label
    return AGE_BAND_RULES[-1][2]


def age_bands_from_years(age_min: object, age_max: object) -> str:
    if pd.isna(age_min) or pd.isna(age_max):
        return ''
    min_month = int(age_min) * 12
    max_month = int(age_max) * 12 + 11
    return join_unique_pipe(
        label
        for band_min, band_max, label in AGE_BAND_RULES
        if not (max_month < band_min or min_month > band_max)
    )


def extract_tag_matches(text: object, pattern_map: dict[str, list[str]]) -> list[str]:
    normalized = normalize_space(text)
    if not normalized:
        return []
    matched = []
    for tag, patterns in pattern_map.items():
        if any(re.search(pattern, normalized, flags=re.I) for pattern in patterns):
            matched.append(tag)
    return matched


def build_place_alias_map(places_df: pd.DataFrame) -> dict[str, set[str]]:
    alias_map: dict[str, set[str]] = {}
    for row in places_df[['place_id', 'place_name']].fillna('').itertuples(index=False):
        place_name = normalize_space(row.place_name)
        candidates = [place_name, re.sub(r'\([^)]*\)', '', place_name).strip()]
        prefix_stripped = re.sub(r'^서울형\s*키즈카페\s*', '', candidates[-1]).strip()
        if prefix_stripped:
            candidates.append(prefix_stripped)
        candidates.extend(re.findall(r'\(([^)]*)\)', place_name))
        alias_map[normalize_space(row.place_id)] = {
            compact_text(candidate)
            for candidate in candidates
            if len(compact_text(candidate)) >= 4
        }
    return alias_map


def strip_review_noise(raw_text: object) -> str:
    text = '' if raw_text is None or pd.isna(raw_text) else str(raw_text)
    text = ZERO_WIDTH_CLUSTER_RE.sub('\n\n', text)
    text = ZERO_WIDTH_RE.sub(' ', text)
    text = MAP_WIDGET_RE.sub('\n\n', text)
    text = VIDEO_WIDGET_RE.sub('\n\n', text)
    for phrase in EXTRA_NOISE_PHRASES:
        text = text.replace(phrase, ' ')
    text = INLINE_URL_RE.sub(' ', text)
    text = INLINE_DOMAIN_RE.sub(' ', text)
    text = HASHTAG_RE.sub(' ', text)
    for marker in BULLET_MARKERS:
        text = text.replace(marker, f'\n\n{marker} ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def split_review_paragraphs(cleaned_text: str) -> list[str]:
    return [normalize_space(paragraph) for paragraph in cleaned_text.split('\n\n') if normalize_space(paragraph)]


def is_noise_paragraph(paragraph: str, title: str) -> bool:
    normalized = normalize_space(paragraph)
    if not normalized:
        return True
    if normalized == normalize_space(title):
        return True
    if re.fullmatch(r'(#[^\s#]+\s*)+', normalized):
        return True
    if normalized.startswith('50m © NAVER Corp'):
        return True
    if '이 블로그의 체크인 이 장소의 다른 글' in normalized:
        return True
    return False


def is_official_info_only_paragraph(paragraph: str) -> bool:
    official_hits = sum(keyword in paragraph for keyword in OFFICIAL_ONLY_KEYWORDS)
    experience_hits = sum(keyword in paragraph for keyword in EXPERIENCE_KEYWORDS)
    return official_hits >= 2 and experience_hits == 0 and len(paragraph) <= 500


def mentions_other_place_only(paragraph: str, current_place_id: str, alias_map: dict[str, set[str]]) -> bool:
    compact_paragraph = compact_text(paragraph)
    current_hits = any(alias in compact_paragraph for alias in alias_map.get(current_place_id, set()))
    for place_id, aliases in alias_map.items():
        if place_id == current_place_id:
            continue
        if any(alias in compact_paragraph for alias in aliases):
            return not current_hits
    return False


def extract_age_metadata(*texts: object) -> dict[str, str]:
    matches: list[tuple[int, str, int]] = []
    joined_text = ' '.join(normalize_space(text) for text in texts if normalize_space(text))
    for pattern, unit in [(MONTH_AGE_RE, 'month'), (YEAR_AGE_RE, 'year')]:
        for match in pattern.finditer(joined_text):
            numeric = int(match.group(1))
            months = numeric if unit == 'month' else numeric * 12
            matches.append((match.start(), match.group(0), months))
    matches.sort(key=lambda item: item[0])
    age_mentions = [raw for _, raw, _ in matches]
    primary_months = matches[0][2] if matches else None
    return {
        'mentioned_age_raw': matches[0][1] if matches else '',
        'mentioned_age_band': age_band_from_months(primary_months),
        'all_age_mentions': join_unique_pipe(age_mentions),
    }


def calculate_doc_quality(raw_content_length: int, clean_content: str, paragraph_count: int, play_tags: str, need_tags: str, issue_tags: str) -> float:
    if raw_content_length <= 0 or not clean_content:
        return 0.0
    removed_ratio = 1 - (len(clean_content) / raw_content_length)
    score = 0.15
    if len(clean_content) >= 400:
        score += 0.20
    if paragraph_count >= 3:
        score += 0.20
    if play_tags or need_tags or issue_tags:
        score += 0.25
    if removed_ratio <= 0.75:
        score += 0.20
    return round(min(score, 1.0), 2)


def split_sentences(text: str) -> list[str]:
    normalized = normalize_space(text)
    if not normalized:
        return []
    sentences = re.split(r'(?<=[.!?])\s+|(?<=다\.)\s+|(?<=요\.)\s+|(?<=죠\.)\s+', normalized)
    sentences = [normalize_space(sentence) for sentence in sentences if normalize_space(sentence)]
    if len(sentences) == 1 and len(normalized) > 500:
        sentences = re.split(r',\s+|·\s*|;\s+', normalized)
        sentences = [normalize_space(sentence) for sentence in sentences if normalize_space(sentence)]
    return sentences or [normalized]


def split_long_text(text: str, target_chars: int = 420, hard_max_chars: int = 520) -> list[str]:
    sentences = split_sentences(text)
    chunks = []
    current = []
    current_length = 0
    for sentence in sentences:
        addition_length = len(sentence) + (1 if current else 0)
        if current and current_length + addition_length > hard_max_chars:
            chunks.append(' '.join(current))
            current = [sentence]
            current_length = len(sentence)
            continue
        current.append(sentence)
        current_length += addition_length
        if current_length >= target_chars:
            chunks.append(' '.join(current))
            current = []
            current_length = 0
    if current:
        chunks.append(' '.join(current))
    return [normalize_space(chunk) for chunk in chunks if normalize_space(chunk)]


def build_review_chunks(clean_content: str, target_chars: int = 420, hard_max_chars: int = 520) -> list[str]:
    units = []
    for paragraph in [paragraph for paragraph in clean_content.split('\n\n') if paragraph]:
        normalized = normalize_space(paragraph)
        if len(normalized) <= hard_max_chars:
            units.append(normalized)
        else:
            units.extend(split_long_text(normalized, target_chars=target_chars, hard_max_chars=hard_max_chars))
    chunks = []
    buffer = []
    buffer_length = 0
    for unit in units:
        addition_length = len(unit) + (2 if buffer else 0)
        if buffer and buffer_length + addition_length > hard_max_chars:
            chunks.append('\n\n'.join(buffer))
            buffer = [unit]
            buffer_length = len(unit)
            continue
        buffer.append(unit)
        buffer_length += addition_length
        if buffer_length >= target_chars:
            chunks.append('\n\n'.join(buffer))
            buffer = []
            buffer_length = 0
    if buffer:
        chunks.append('\n\n'.join(buffer))
    return [chunk for chunk in chunks if normalize_space(chunk)]


def classify_chunk_section(play_tags: str, need_tags: str, issue_tags: str) -> str:
    if issue_tags:
        return 'warning'
    if play_tags:
        return 'play_experience'
    if need_tags:
        return 'access_or_parent_context'
    return 'general_review'


def build_chunk_embedding_text(place_name: str, review_title: str, chunk_text: str, mentioned_age_raw: str, play_tags: str, need_tags: str, issue_tags: str) -> str:
    parts = [normalize_space(place_name)]
    if review_title:
        parts.append(f'리뷰 제목: {normalize_space(review_title)}')
    if mentioned_age_raw:
        parts.append(f'연령 단서: {mentioned_age_raw}')
    if play_tags:
        parts.append(f'놀이 태그: {play_tags}')
    if need_tags:
        parts.append(f'이용 맥락 태그: {need_tags}')
    if issue_tags:
        parts.append(f'주의 태그: {issue_tags}')
    parts.append(f'리뷰 내용: {normalize_space(chunk_text)}')
    return '. '.join(part for part in parts if part)


def build_official_play_tags(place_row: dict[str, object], feature_row: dict[str, object]) -> str:
    texts = [place_row.get('program_info_text', ''), place_row.get('usage_rules_text', ''), place_row.get('usage_age_detail_text', '')]
    tags = extract_tag_matches(' '.join(normalize_space(text) for text in texts), PLAY_TAG_PATTERNS)
    if normalize_space(feature_row.get('program_role_play')) == 'yes':
        tags.append('역할놀이')
    if normalize_space(feature_row.get('program_media_play')) == 'yes':
        tags.append('미디어체험')
    if normalize_space(feature_row.get('program_active_play')) == 'yes':
        tags.append('대근육')
    if normalize_space(feature_row.get('program_creative_play')) == 'yes':
        tags.append('창의활동')
    return join_unique_pipe(tags)


def build_official_service_tags(place_row: dict[str, object], feature_row: dict[str, object]) -> str:
    tags = []
    parking_info = normalize_space(place_row.get('parking_info'))
    operating_day_text = normalize_space(place_row.get('operating_day_text'))
    if normalize_space(feature_row.get('parking_available')) == 'yes' or parking_info:
        tags.append('주차가능')
    if normalize_space(feature_row.get('parking_paid')) == 'yes' or '주차요금' in parking_info:
        tags.append('유료주차')
    if normalize_space(feature_row.get('reservation_available')) == 'yes' or normalize_space(place_row.get('reservation_url')):
        tags.append('사전예약')
    if normalize_space(feature_row.get('weekend_operation')) == 'yes' or ('토' in operating_day_text or '일' in operating_day_text):
        tags.append('주말운영')
    if normalize_space(feature_row.get('care_service_available')) == 'yes':
        tags.append('돌봄서비스')
    if normalize_space(feature_row.get('group_visit_available')) == 'yes':
        tags.append('단체이용')
    if normalize_space(feature_row.get('guardian_required')) == 'yes':
        tags.append('보호자동반')
    if normalize_space(feature_row.get('toddler_friendly')) == 'yes':
        tags.append('영아친화')
    if normalize_space(feature_row.get('preschool_friendly')) == 'yes':
        tags.append('유아친화')
    if normalize_space(feature_row.get('lower_elementary_friendly')) == 'yes':
        tags.append('초등저친화')
    parking_difficulty = normalize_space(feature_row.get('parking_difficulty'))
    if parking_difficulty == 'easy':
        tags.append('주차쉬움')
    elif parking_difficulty == 'limited':
        tags.append('주차제한')
    return join_unique_pipe(tags)


def build_official_summary(place_row: dict[str, object]) -> str:
    parts = [normalize_space(place_row.get('place_name'))]
    if normalize_space(place_row.get('facility_type')):
        parts.append(f"유형 {normalize_space(place_row.get('facility_type'))}")
    if normalize_space(place_row.get('age_text')):
        parts.append(f"이용 연령 {normalize_space(place_row.get('age_text'))}")
    if normalize_space(place_row.get('official_play_tags')):
        parts.append(f"주요 놀이 {normalize_space(place_row.get('official_play_tags')).replace('|', ', ')}")
    if normalize_space(place_row.get('official_service_tags')):
        parts.append(f"운영/편의 {normalize_space(place_row.get('official_service_tags')).replace('|', ', ')}")
    if normalize_space(place_row.get('address_gu')):
        parts.append(f"위치 {normalize_space(place_row.get('address_gu'))}")
    return '. '.join(part for part in parts if part)


RAW_PLACES_PATH, RAW_REVIEW_DOCS_PATH, RAW_PLACE_FEATURES_PATH


(PosixPath('/Users/jkyu/Desktop/ongoing/BebeNori/01_data_prep/outputs/raw/places.csv'),
 PosixPath('/Users/jkyu/Desktop/ongoing/BebeNori/01_data_prep/outputs/raw/review_docs.csv'),
 PosixPath('/Users/jkyu/Desktop/ongoing/BebeNori/01_data_prep/outputs/raw/place_features.csv'))

In [2]:
places_raw_df = pd.read_csv(RAW_PLACES_PATH)
review_docs_raw_df = pd.read_csv(RAW_REVIEW_DOCS_PATH)
place_features_raw_df = pd.read_csv(RAW_PLACE_FEATURES_PATH)

print('places_raw_df', places_raw_df.shape)
print('review_docs_raw_df', review_docs_raw_df.shape)
print('place_features_raw_df', place_features_raw_df.shape)


places_raw_df (124, 27)
review_docs_raw_df (766, 5)
place_features_raw_df (124, 25)


In [3]:
# place_features는 빠른 필터링/랭킹용 정형 테이블이므로 짧고 안정적인 값만 유지합니다.
place_features_df = place_features_raw_df.copy()
if 'district' in place_features_df.columns and 'address_gu' not in place_features_df.columns:
    place_features_df = place_features_df.rename(columns={'district': 'address_gu'})
if 'address_dong' not in place_features_df.columns:
    address_dong_map = places_raw_df.set_index('place_id')['address'].apply(extract_address_dong).to_dict()
    place_features_df['address_dong'] = place_features_df['place_id'].map(address_dong_map).fillna('')

expected_feature_columns = [
    'place_id', 'address_gu', 'address_dong', 'parking_available', 'parking_paid', 'reservation_available',
    'toddler_friendly', 'preschool_friendly', 'lower_elementary_friendly', 'weekend_operation',
    'price_level', 'discount_available', 'care_service_available', 'guardian_required',
    'parking_difficulty', 'group_visit_available', 'program_role_play', 'program_media_play',
    'safety_negative', 'food_allowed', 'program_active_play', 'program_creative_play',
    'cleanliness_positive', 'spacious_positive', 'crowded_warning',
]
for column_name in expected_feature_columns:
    if column_name not in place_features_df.columns:
        place_features_df[column_name] = ''
    place_features_df[column_name] = place_features_df[column_name].apply(lambda value: normalize_feature_value(column_name, value))
place_features_df = place_features_df[expected_feature_columns]

# places는 공식 원문 마스터이므로, 리뷰 없는 매장 추천에도 쓸 수 있게 공식 태그/요약을 추가합니다.
places_df = places_raw_df.copy()
if 'district' in places_df.columns and 'address_gu' not in places_df.columns:
    places_df = places_df.rename(columns={'district': 'address_gu'})
if 'car_info' in places_df.columns:
    places_df['parking_info'] = [merge_unique_text(parking_info, car_info) for parking_info, car_info in zip(places_df['parking_info'], places_df['car_info'])]
    places_df = places_df.drop(columns=['car_info'])
for column_name in places_df.columns:
    if places_df[column_name].dtype == object:
        places_df[column_name] = places_df[column_name].apply(normalize_space)
for column_name in ['capacity_personal', 'capacity_group', 'age_min', 'age_max']:
    if column_name in places_df.columns:
        places_df[column_name] = pd.to_numeric(places_df[column_name], errors='coerce').astype('Int64')
if 'address_dong' not in places_df.columns:
    places_df['address_dong'] = places_df['address'].apply(extract_address_dong)

feature_map = place_features_df.set_index('place_id').to_dict(orient='index')
enriched_places = []
for place_row in places_df.to_dict(orient='records'):
    feature_row = feature_map.get(place_row['place_id'], {})
    place_row['official_age_bands'] = age_bands_from_years(place_row.get('age_min'), place_row.get('age_max'))
    place_row['official_play_tags'] = build_official_play_tags(place_row, feature_row)
    place_row['official_service_tags'] = build_official_service_tags(place_row, feature_row)
    place_row['official_summary'] = build_official_summary(place_row)
    enriched_places.append(place_row)
places_df = pd.DataFrame(enriched_places)

place_features_df.head(), places_df.head()


(   place_id address_gu address_dong parking_available parking_paid  \
 0  db231001        도봉구          도봉동               yes                
 1  db231201        도봉구          방학동                                  
 2  db231202        도봉구           창동               yes                
 3  db241101        도봉구           창동               yes          yes   
 4  db241102        도봉구          도봉동               yes           no   
 
   reservation_available toddler_friendly preschool_friendly  \
 0                                    yes                yes   
 1                   yes              yes                yes   
 2                   yes              yes                yes   
 3                   yes              yes                yes   
 4                   yes              yes                yes   
 
   lower_elementary_friendly weekend_operation  ... group_visit_available  \
 0                                              ...                   yes   
 1                              

In [4]:
# preprocessed/review_docs.csv 는 문서가 아니라 chunk 단위 최종 산출물입니다.
# raw/review_docs.csv 원본은 유지하고, 여기서는 RAG 검색에 바로 쓸 수 있는 행 단위만 만듭니다.
review_docs_source_df = review_docs_raw_df.copy()
if 'source_url' not in review_docs_source_df.columns and 'doc_url' in review_docs_source_df.columns:
    review_docs_source_df = review_docs_source_df.rename(columns={'doc_url': 'source_url'})
for column_name in ['place_id', 'place_name', 'title', 'content', 'source_url']:
    if column_name not in review_docs_source_df.columns:
        review_docs_source_df[column_name] = ''
review_docs_source_df = review_docs_source_df[['place_id', 'place_name', 'title', 'content', 'source_url']]

alias_map = build_place_alias_map(places_df)
processed_review_rows = []
seen_source_urls = set()
seen_content_hashes = set()

for raw_row in review_docs_source_df.to_dict(orient='records'):
    place_id = normalize_space(raw_row.get('place_id'))
    place_name = normalize_space(raw_row.get('place_name'))
    review_title = normalize_space(raw_row.get('title'))
    source_url = normalize_space(raw_row.get('source_url'))
    raw_content = '' if pd.isna(raw_row.get('content')) else str(raw_row.get('content'))

    stripped_text = strip_review_noise(raw_content)
    paragraphs = split_review_paragraphs(stripped_text)
    non_noise_paragraphs = [
        paragraph
        for paragraph in paragraphs
        if not is_noise_paragraph(paragraph, review_title)
        and not mentions_other_place_only(paragraph, place_id, alias_map)
    ]
    kept_paragraphs = [paragraph for paragraph in non_noise_paragraphs if not is_official_info_only_paragraph(paragraph)]

    # 리뷰 원문이 공식 정보 위주여도 최소한의 근거는 남기기 위해 한 단계 되돌립니다.
    if not kept_paragraphs:
        kept_paragraphs = non_noise_paragraphs

    clean_content = '\n\n'.join(kept_paragraphs)
    if not clean_content:
        continue

    dedupe_hash = hashlib.sha1(f"{place_id}|{compact_text(review_title)}|{compact_text(clean_content)}".encode('utf-8')).hexdigest()
    if source_url and source_url in seen_source_urls:
        continue
    if dedupe_hash in seen_content_hashes:
        continue
    seen_source_urls.add(source_url)
    seen_content_hashes.add(dedupe_hash)

    age_metadata = extract_age_metadata(review_title, clean_content)
    play_tags = join_unique_pipe(extract_tag_matches(clean_content, PLAY_TAG_PATTERNS))
    need_tags = join_unique_pipe(extract_tag_matches(clean_content, NEED_TAG_PATTERNS))
    issue_tags = join_unique_pipe(extract_tag_matches(clean_content, ISSUE_TAG_PATTERNS))
    doc_quality_score = calculate_doc_quality(
        raw_content_length=len(raw_content),
        clean_content=clean_content,
        paragraph_count=len(kept_paragraphs),
        play_tags=play_tags,
        need_tags=need_tags,
        issue_tags=issue_tags,
    )
    doc_id = hashlib.sha1(f"{place_id}|{source_url}|{review_title}".encode('utf-8')).hexdigest()[:12]

    for chunk_order, chunk_text in enumerate(build_review_chunks(clean_content), start=1):
        chunk_age_metadata = extract_age_metadata(review_title, chunk_text)
        chunk_play_tags = join_unique_pipe(extract_tag_matches(chunk_text, PLAY_TAG_PATTERNS)) or play_tags
        chunk_need_tags = join_unique_pipe(extract_tag_matches(chunk_text, NEED_TAG_PATTERNS)) or need_tags
        chunk_issue_tags = join_unique_pipe(extract_tag_matches(chunk_text, ISSUE_TAG_PATTERNS)) or issue_tags
        processed_review_rows.append({
            'chunk_id': f'{doc_id}_c{chunk_order:02d}',
            'doc_id': doc_id,
            'place_id': place_id,
            'place_name': place_name,
            'review_title': review_title,
            'chunk_order': chunk_order,
            'chunk_text': chunk_text,
            'chunk_text_for_embedding': build_chunk_embedding_text(
                place_name=place_name,
                review_title=review_title,
                chunk_text=chunk_text,
                mentioned_age_raw=chunk_age_metadata['mentioned_age_raw'] or age_metadata['mentioned_age_raw'],
                play_tags=chunk_play_tags,
                need_tags=chunk_need_tags,
                issue_tags=chunk_issue_tags,
            ),
            'section_type': classify_chunk_section(chunk_play_tags, chunk_need_tags, chunk_issue_tags),
            'chunk_length': len(chunk_text),
            'mentioned_age_raw': chunk_age_metadata['mentioned_age_raw'] or age_metadata['mentioned_age_raw'],
            'mentioned_age_band': chunk_age_metadata['mentioned_age_band'] or age_metadata['mentioned_age_band'],
            'all_age_mentions': age_metadata['all_age_mentions'],
            'play_tags': chunk_play_tags,
            'need_tags': chunk_need_tags,
            'issue_tags': chunk_issue_tags,
            'source_url': source_url,
            'source_domain': extract_source_domain(source_url),
            'doc_quality_score': doc_quality_score,
        })

review_docs_df = pd.DataFrame(processed_review_rows)
review_docs_df.head()


,chunk_id,doc_id,place_id,place_name,review_title,chunk_order,chunk_text,chunk_text_for_embedding,section_type,chunk_length,mentioned_age_raw,mentioned_age_band,all_age_mentions,play_tags,need_tags,issue_tags,source_url,source_domain,doc_quality_score
0,02aa2422a734_c01,02aa2422a734,DJ230901,서울형 키즈카페 시립 1호점,서울형 키즈카페 시립1호점 후기｜대방역 동작구 실내놀이터 평일 방문 꿀팁,1,간지가 방학이라 조카 일찍 하원시키고 다녀왔어요! 주말은 당일 예약 거의 불가라고 ...,서울형 키즈카페 시립 1호점. 리뷰 제목: 서울형 키즈카페 시립1호점 후기｜대방역 ...,warning,431,,,,대근육|소근육|역할놀이|창의활동|사회성놀이|낚시놀이,주차편의|대중교통편의|예약치열,안전주의,https://m.blog.naver.com/hanc8888/224195023002,m.blog.naver.com,1.0
1,02aa2422a734_c02,02aa2422a734,DJ230901,서울형 키즈카페 시립 1호점,서울형 키즈카페 시립1호점 후기｜대방역 동작구 실내놀이터 평일 방문 꿀팁,2,"🔸 이용요금 : 아동 1인 5,000원\n\n🔸 온라인 사전예약 필수\n\n🔸 현장...",서울형 키즈카페 시립 1호점. 리뷰 제목: 서울형 키즈카페 시립1호점 후기｜대방역 ...,warning,426,,,,창의활동,청결|넓음,안전주의,https://m.blog.naver.com/hanc8888/224195023002,m.blog.naver.com,1.0
2,02aa2422a734_c03,02aa2422a734,DJ230901,서울형 키즈카페 시립 1호점,서울형 키즈카페 시립1호점 후기｜대방역 동작구 실내놀이터 평일 방문 꿀팁,3,✔ 연령대가 다양하게 놀 수 있고\n\n✔ 미끄럼틀이 다른 지점보다 조금 더 스릴있...,서울형 키즈카페 시립 1호점. 리뷰 제목: 서울형 키즈카페 시립1호점 후기｜대방역 ...,warning,443,,,,대근육|소근육|역할놀이|창의활동|사회성놀이|낚시놀이,주차편의|대중교통편의|예약치열|보호자휴식|청결|넓음,안전주의,https://m.blog.naver.com/hanc8888/224195023002,m.blog.naver.com,1.0
3,02aa2422a734_c04,02aa2422a734,DJ230901,서울형 키즈카페 시립 1호점,서울형 키즈카페 시립1호점 후기｜대방역 동작구 실내놀이터 평일 방문 꿀팁,4,명절 시즌이라 줄다리기 이벤트도 진행 중이었는데 아이들끼리 모여서 참여하니 부모님들...,서울형 키즈카페 시립 1호점. 리뷰 제목: 서울형 키즈카페 시립1호점 후기｜대방역 ...,warning,248,,,,사회성놀이,보호자휴식,안전주의,https://m.blog.naver.com/hanc8888/224195023002,m.blog.naver.com,1.0
4,193f9e3edad5_c01,193f9e3edad5,DJ230901,서울형 키즈카페 시립 1호점,서울베이비앰배서더 3기 발대식 및 2기 수료식 🏆 내가 서울시 표창을 받다니!,1,서울베이비앰배서더 3기 발대식 및 2기 수료식 2026년 2월 23일 월요일 작년 ...,서울형 키즈카페 시립 1호점. 리뷰 제목: 서울베이비앰배서더 3기 발대식 및 2기 ...,play_experience,443,24개월,13-24개월,24개월|6개월,역할놀이|사회성놀이,주차편의|예약여유|보호자휴식|형제동반,,https://m.blog.naver.com/kimjiy8/224199423267,m.blog.naver.com,1.0


In [5]:
places_df = enrich_place_coordinates(places_df)

places_df.to_csv(OUTPUT_PLACES_PATH, index=False, encoding='utf-8-sig')
review_docs_df.to_csv(OUTPUT_REVIEW_DOCS_PATH, index=False, encoding='utf-8-sig')
place_features_df.to_csv(OUTPUT_PLACE_FEATURES_PATH, index=False, encoding='utf-8-sig')

summary_df = pd.DataFrame([
    {
        'table_name': 'places',
        'rows': len(places_df),
        'columns': len(places_df.columns),
        'output_path': str(OUTPUT_PLACES_PATH),
    },
    {
        'table_name': 'review_docs',
        'rows': len(review_docs_df),
        'columns': len(review_docs_df.columns),
        'output_path': str(OUTPUT_REVIEW_DOCS_PATH),
    },
    {
        'table_name': 'place_features',
        'rows': len(place_features_df),
        'columns': len(place_features_df.columns),
        'output_path': str(OUTPUT_PLACE_FEATURES_PATH),
    },
])

summary_df


,table_name,rows,columns,output_path
0,places,124,32,/Users/jkyu/Desktop/ongoing/BebeNori/01_data_p...
1,review_docs,2916,19,/Users/jkyu/Desktop/ongoing/BebeNori/01_data_p...
2,place_features,124,25,/Users/jkyu/Desktop/ongoing/BebeNori/01_data_p...
